In [ ]:
def avaliacao(df, modelo, metrica):

    # Listas de métricas e modelos disponíveis
    lista_modelos = ['media', 'mediana', 'knn']
    lista_metricas = ['RMSE', 'MAE', 'bias', 'r2']

    # Busca a presença de NaN no DataFrame
    if df.columns[df.isna().any()].array.size != 0:
    # Remove linhas com NaN, caso elas existam
        df = df.dropna().copy()
    # Mantém df, caso ele já não apresente NaN
    else:
        df = df.copy()   

    # Armazena as listas de metadados e variáveis de df
    from utils import separar_colunas
    metadados, variaveis = separar_colunas(df)

    # Verifica se a métrica e modelo especificados na função estão dentro das listas
    if metrica not in lista_metricas:
        raise ValueError("metrica deve conter uma das seguinte 4 métricas: 'RMSE', 'MAE', 'bias', 'r2'")
    elif modelo not in lista_modelos:
        raise ValueError("modelo deve conter um dos seguinte 3 modelos: 'media', 'mediana', 'KNN'")
    else:
        pass

    if modelo == 'media':
        from imputacao import mean_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = mean_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = mean_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo == 'mediana':
        from imputacao import median_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.05, metadados=metadados)
        df_artificial_parcial = median_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = median_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    elif modelo.lower() == 'knn':
        from imputacao import knn_imput
        df_artificial_parcial, df_artificial_total = criar_validacao_cruzada(df, pct_remover=0.15, metadados=metadados)
        df_artificial_parcial = knn_imput(df_artificial_parcial, metadados=metadados, variaveis=variaveis, return_reduced=True)
        df_artificial_total = knn_imput(df_artificial_total, metadados=metadados, variaveis=variaveis, return_reduced=True)

    if metrica.lower() == 'rmse':
        from sklearn.metrics import root_mean_squared_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return root_mean_squared_error(df_verd, df_art)
    
    elif metrica.lower() == 'mae':
        from sklearn.metrics import mean_absolute_error

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return mean_absolute_error(df_verd, df_art)
    
    elif metrica == 'r2':
        from sklearn.metrics import r2_score

        df_verd, df_art, colunas =  preparar_para_metricas(df, df_artificial_parcial, metadados)

        return r2_score(df_verd, df_art)

In [ ]:
# Teste
df_nan_5, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.05,  # Remove 5% --> Imputação via média e mediana
    metadados=metadados
)

df_nan_15, dados_sim_a, dados_sim_b, cols_nan = criar_validacao_cruzada(
    dados_limpos, 
    pct_remover=0.15,  # Remove 15% --> Imputação via KNN
    metadados=metadados
)

from imputacao import mean_imput, median_imput, knn_imput

# Apenas colunas com NaN originalmente
imput_mean_a = mean_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_a = median_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_a = knn_imput(dados_sim_a, metadados=metadados, variaveis=variaveis, return_reduced=True)

# DataFrame completo
imput_mean_b = mean_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_median_b = median_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)
imput_knn_b = knn_imput(dados_sim_b, metadados=metadados, variaveis=variaveis, return_reduced=True)